# Detector de fugas de gas en video OGI

Generación de penachos sintéticos para entrenar un detector sin tener que
etiquetar video real a mano.

Ejecuta las celdas en orden con **Shift + Enter**, o desde el menú
*Entorno de ejecución → Ejecutar todas*.

## 1. Descargar el proyecto

In [ ]:
!git clone -q https://github.com/franyer98/ogi-leak-detector
%cd ogi-leak-detector
!ls

## 2. Una escena sin gas

Así se ve la escena base: tanques, tubería, vegetación y el grano del sensor.

In [ ]:
from fondo import escena_ogi
from google.colab.patches import cv2_imshow

fondo, accesorios = escena_ogi(256, 448, semilla=11)
print('accesorios detectados (posibles puntos de fuga):', accesorios)
cv2_imshow(fondo)

## 3. Agregar el penacho

El mismo fondo, ahora con una fuga saliendo de un accesorio del techo.

In [ ]:
from penacho import Penacho, componer, caja
import cv2

p = Penacho(256, 448, origen=accesorios[0], intensidad=0.8,
            viento=0.3, semilla=5)
mascara = p.mascara(0.8)
imagen = componer(fondo, mascara, 'oscuro', 0.66)

vis = cv2.cvtColor(imagen, cv2.COLOR_GRAY2BGR)
b = caja(mascara)
cv2.rectangle(vis, (b[0], b[1]), (b[2], b[3]), (0, 220, 255), 1)
cv2_imshow(vis)
print('caja del penacho:', b)

## 4. La intensidad refleja el caudal

Una fuga fuerte sube más rápido, llega más alto y se ve más densa.

In [ ]:
import numpy as np

tiras = []
for inten in [0.25, 0.55, 0.9]:
    pp = Penacho(256, 448, origen=accesorios[0], intensidad=inten,
                 viento=0.3, semilla=5)
    img = componer(fondo, pp.mascara(0.8), 'oscuro', 0.66)
    v = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    cv2.putText(v, f'intensidad {inten}', (8, 18),
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 220, 255), 1)
    tiras.append(v)

cv2_imshow(np.hstack(tiras))

## 5. El penacho se mueve

Cuatro instantes de la misma fuga: la turbulencia lo deforma y el viento
lo arrastra. Ese movimiento es lo que distingue al gas de una mancha fija.

In [ ]:
tiras = []
for t in [0, 0.4, 0.8, 1.2]:
    img = componer(fondo, p.mascara(t), 'oscuro', 0.66)
    tiras.append(cv2.cvtColor(img, cv2.COLOR_GRAY2BGR))

cv2_imshow(np.hstack(tiras))

## 6. Generar el conjunto de entrenamiento

Cada fotograma sale con su etiqueta ya calculada. Incluye secuencias sin
fuga: sin ellas el detector marcaría penachos donde solo hay temblor de
cámara.

Sube `n_secuencias` para un conjunto más grande.

In [ ]:
from dataset import construir

stats = construir(n_secuencias=60, n_frames=10)
print(stats)

## 7. Revisar las etiquetas generadas

In [ ]:
import glob, random
random.seed(7)

archivos = sorted(glob.glob('dataset/images/train/*.png'))
con, sin = [], []
for a in archivos:
    etiqueta = a.replace('/images/', '/labels/').replace('.png', '.txt')
    (con if open(etiqueta).read().strip() else sin).append(a)

print(f'fotogramas con fuga: {len(con)}  |  sin fuga: {len(sin)}')

tiras = []
for a in random.sample(con, 3) + random.sample(sin, 1):
    img = cv2.imread(a, cv2.IMREAD_GRAYSCALE)
    v = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    H, W = img.shape
    etiqueta = a.replace('/images/', '/labels/').replace('.png', '.txt')
    n = 0
    for linea in open(etiqueta):
        _, cx, cy, w, h = map(float, linea.split())
        cv2.rectangle(v, (int((cx - w/2) * W), int((cy - h/2) * H)),
                         (int((cx + w/2) * W), int((cy + h/2) * H)), (0, 220, 255), 1)
        n += 1
    cv2.putText(v, f'{n} fuga(s)', (8, 18), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 220, 255), 1)
    tiras.append(v)

cv2_imshow(np.hstack(tiras))

## 8. Descargar el conjunto

Queda comprimido para usarlo en otro entorno.

In [ ]:
!zip -q -r dataset.zip dataset
from google.colab import files
files.download('dataset.zip')

---

## Siguiente paso: entrenar el detector

Con el conjunto ya generado, el entrenamiento se puede hacer aquí mismo.
Antes conviene activar la GPU: *Entorno de ejecución → Cambiar tipo de
entorno → GPU*.

```python
!pip install -q ultralytics
from ultralytics import YOLO

modelo = YOLO('yolov8n.pt')
modelo.train(data='dataset/data.yaml', epochs=40, imgsz=448)
```

Un detalle importante: un modelo entrenado solo con datos sintéticos
necesita validarse contra video real antes de darlo por bueno. Los
penachos generados se parecen a los reales, pero no son idénticos.